# 04 — Streaming Analytics

Queries the Silver table to show real-time gaming floor analytics.
These are the same queries we'll run in Part 2 (Zerobus) for an
apples-to-apples comparison.

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

SILVER_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_silver"

## Revenue by Casino Floor

How much is each floor generating? (House revenue = total bets - total wins)

In [ ]:
spark.sql(f"""
    SELECT
        casino_floor,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_bets,
        ROUND(SUM(win_amount), 2) AS total_payouts,
        ROUND(SUM(bet_amount) - SUM(win_amount), 2) AS house_revenue
    FROM {SILVER_TABLE}
    GROUP BY casino_floor
    ORDER BY house_revenue DESC
""").show(truncate=False)

## Top 10 Machines by Play Volume

In [ ]:
spark.sql(f"""
    SELECT
        machine_id,
        casino_floor,
        game_type,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_wagered
    FROM {SILVER_TABLE}
    GROUP BY machine_id, casino_floor, game_type
    ORDER BY total_plays DESC
    LIMIT 10
""").show(truncate=False)

## Win/Loss Distribution by Game Type

In [ ]:
spark.sql(f"""
    SELECT
        game_type,
        COUNT(*) AS total_plays,
        SUM(CASE WHEN is_win THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN NOT is_win THEN 1 ELSE 0 END) AS losses,
        ROUND(SUM(CASE WHEN is_win THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_pct,
        ROUND(AVG(bet_amount), 2) AS avg_bet,
        ROUND(AVG(CASE WHEN is_win THEN win_amount END), 2) AS avg_win_payout
    FROM {SILVER_TABLE}
    GROUP BY game_type
    ORDER BY total_plays DESC
""").show(truncate=False)

## Bet Tier Breakdown

How does play behavior break down across Low / Medium / High / Whale tiers?

In [ ]:
spark.sql(f"""
    SELECT
        bet_tier,
        COUNT(*) AS total_plays,
        COUNT(DISTINCT player_id) AS unique_players,
        ROUND(SUM(bet_amount), 2) AS total_wagered,
        ROUND(SUM(bet_amount) - SUM(win_amount), 2) AS house_revenue
    FROM {SILVER_TABLE}
    GROUP BY bet_tier
    ORDER BY total_wagered DESC
""").show(truncate=False)